# Dataset audit

## Mục tiêu
Data engineer giữ raw bất biến, xác nhận quyền sử dụng, kiểm tra nhãn và tách người/session. Bộ mẫu này tự sinh, không chứa người thật và không dùng để kết luận chất lượng model.

## Setup
Chạy từ repo đã clone ở revision bạn ghi nhận. Local dùng venv API; Colab xem docs/RESEARCH.md. Không tự cài dependency hoặc tải dữ liệu khi Run all.

In [1]:
from pathlib import Path
import os, sys, json, tempfile
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai/research").is_dir()), None)
assert ROOT is not None, "Clone repo và chạy từ repo/notebooks"
sys.path.insert(0, str(ROOT))
os.environ["YOLO_AUTOINSTALL"] = "false"
from ai.research.cli import environment
print(json.dumps(environment(), indent=2))

{
  "python": "3.11.9",
  "platform": "Windows-10-10.0.26200-SP0",
  "versions": {
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "ultralytics": "8.4.162",
    "numpy": "1.26.4",
    "pillow": "10.3.0"
  },
  "cuda_available": true,
  "gpu": "NVIDIA GeForce RTX 3060"
}


## Các bước
Đọc cấu hình trước khi thực thi; các thao tác tốn tài nguyên mặc định tắt.

In [2]:
from PIL import Image
from ai.research.data import inventory, release
with tempfile.TemporaryDirectory() as directory:
    root = Path(directory); raw = root / "raw"; labels = root / "labels"
    raw.mkdir(); labels.mkdir()
    Image.new("RGB", (64, 64), "red").save(raw / "fixture.png")
    labels.joinpath("fixture.txt").write_text("0 0.5 0.5 1 1 " + " ".join(["0.5 0.5 2"] * 17))
    blocked = inventory(raw)
    assert blocked["images"][0]["status"] == "quarantine"
    row = {"path": "fixture.png", "source": "generated tutorial fixture", "rights_ref": "generated-no-person",
           "training_allowed": True, "annotation_reviewed": True, "annotation_version": "fixture-v1",
           "subject_id": "fixture", "session_id": "fixture-session"}
    meta = root / "metadata.jsonl"; meta.write_text(json.dumps(row), encoding="utf-8")
    report = inventory(raw, meta)
    dataset = release(report, labels, root / "release", smoke=True)
    assert dataset["smoke_only"] is True
    print({"without_rights": "quarantine", "release_images": len(dataset["images"]), "smoke_only": True,
           "near_duplicate_groups": len(report["near_duplicate_review_groups"])})

{'without_rights': 'quarantine', 'release_images': 1, 'smoke_only': True, 'near_duplicate_groups': 0}


## Kiểm tra
Output ghi rõ thao tác thực sự chạy và thao tác bị bỏ qua. Thiếu dữ liệu không được thay bằng số giả. Khi thay dữ liệu/cấu hình, restart kernel và Run all.

## Bước tiếp theo
Import ảnh của bạn bằng CLI inventory; review annotation và nhóm near-duplicate trước release. Keypoints không thay thế segmentation masks.